In [1]:
# Install required packages
!pip install -q transformers datasets seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
# Parse CoNLL format
def read_conll(file_path):
    sentences = []
    labels = []
    with open(file_path, encoding='utf-8') as f:
        tokens = []
        tags = []
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
            else:
                token, tag = line.split()
                tokens.append(token)
                tags.append(tag)
    return sentences, labels

# Replace this path with your uploaded file path
file_path = "/content/amharic_ner_conll_format.txt"
sentences, tags = read_conll(file_path)

In [4]:
# Convert to Hugging Face Dataset format
from datasets import Dataset

dataset = Dataset.from_dict({
    "tokens": sentences,
    "ner_tags": tags
})
label_list = list(set(tag for seq in tags for tag in seq))
label_list.sort()
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [6]:
# Tokenize and align labels
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Davlan/xlm-roberta-base-finetuned-amharic")

def tokenize_and_align(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True, padding=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            else:
                label_str = label[word_idx]
                if word_idx != prev_word_idx:
                    label_ids.append(label2id[label_str])
                else:
                    if label_str.startswith("B-"):
                        label_str = label_str.replace("B-", "I-")
                    label_ids.append(label2id[label_str])
                prev_word_idx = word_idx
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align, batched=True)

tokenizer_config.json:   0%|          | 0.00/356 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

In [7]:
# Fine-tune model
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(
    "Davlan/xlm-roberta-base-finetuned-amharic",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./ner_model",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    save_steps=10_000,
    logging_dir="./logs",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

trainer.train()

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at Davlan/xlm-roberta-base-finetuned-amharic and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-7-1718999729.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: zumihibet2 (zumihibet2-addis-ababa-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss


TrainOutput(global_step=21, training_loss=0.5348872684297108, metrics={'train_runtime': 649.6306, 'train_samples_per_second': 0.226, 'train_steps_per_second': 0.032, 'total_flos': 18981278970192.0, 'train_loss': 0.5348872684297108, 'epoch': 3.0})

In [8]:
trainer.save_model("/content/amharic-ner-xlmamh")
tokenizer.save_pretrained("/content/amharic-ner-xlmamh")

('/content/amharic-ner-xlmamh/tokenizer_config.json',
 '/content/amharic-ner-xlmamh/special_tokens_map.json',
 '/content/amharic-ner-xlmamh/sentencepiece.bpe.model',
 '/content/amharic-ner-xlmamh/added_tokens.json',
 '/content/amharic-ner-xlmamh/tokenizer.json')